# LoveDA Dataset Exploration

This notebook explores the LoveDA aerial imagery dataset.

**LoveDA** (Land-cOver Domain Adaptive semantic segmentation):
- 7 semantic classes: Background, Building, Road, Water, Barren, Forest, Agriculture
- Urban and Rural domain splits
- ~5000 1024×1024 RGB aerial images at 0.3m GSD resolution

**Contents:**
1. Dataset structure and statistics
2. Class distribution visualisation
3. Sample image-mask pairs
4. Domain comparison (Urban vs Rural)

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from pathlib import Path

from src.data.dataset import LoveDA

sns.set_theme(style='darkgrid')
%matplotlib inline

In [ ]:
DATA_ROOT = '../data'

train_dataset = LoveDA(root=DATA_ROOT, split='train', transform=None)
val_dataset = LoveDA(root=DATA_ROOT, split='val', transform=None)
test_dataset = LoveDA(root=DATA_ROOT, split='test', transform=None)

print(f'Train: {len(train_dataset)} images')
print(f'Val:   {len(val_dataset)} images')
print(f'Test:  {len(test_dataset)} images (no masks)')

In [ ]:
# Class distribution
dist = train_dataset.get_class_distribution()
print('Class pixel distribution (training set):')
for name, freq in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {name:15s}: {freq*100:.2f}%')

In [ ]:
# Visualise sample image-mask pairs
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i in range(8):
    image, mask = train_dataset[i]
    
    img_np = image.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)
    mask_np = mask.numpy().astype(np.uint8)
    mask_rgb = LoveDA.decode_mask(mask_np)
    
    axes[i * 2].imshow(img_np)
    axes[i * 2].set_title(f'Image {i}', fontsize=10)
    axes[i * 2].axis('off')
    
    axes[i * 2 + 1].imshow(mask_rgb)
    axes[i * 2 + 1].set_title(f'Mask {i}', fontsize=10)
    axes[i * 2 + 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Urban vs Rural domain comparison
urban_train = LoveDA(root=DATA_ROOT, split='train', domains=['Urban'], transform=None)
rural_train = LoveDA(root=DATA_ROOT, split='train', domains=['Rural'], transform=None)

urban_dist = urban_train.get_class_distribution()
rural_dist = rural_train.get_class_distribution()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, dist) in zip(axes, [('Urban', urban_dist), ('Rural', rural_dist)]):
    names = list(dist.keys())
    values = list(dist.values())
    bars = ax.bar(names, values, color=sns.color_palette('husl', 7))
    ax.set_title(f'{title} Domain Class Distribution', fontsize=12)
    ax.set_ylabel('Pixel Frequency')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val*100:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()